# Configuration

In [1]:
import sys
import code
import os
import pickle
import scanpy as sc
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np

from tqdm import tqdm
import scipy.sparse
from scipy.sparse import csr_matrix
from gtfparse import read_gtf
from collections import defaultdict
from sklearn.metrics import auc, precision_recall_curve, average_precision_score

sys.path.append('/home/wuyan/dygmamba_project/model/dygmamba/src/')

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2



In [3]:
data_root= "/home/wuyan/dygmamba_project/"

cell_type = "H1"

output_path = data_root + "data/cell_line/" + cell_type + "/process/"

print("\n ********************---process rna data---***************************** \n")

adata_rna = ad.read_h5ad(output_path + "rna_origin.h5ad")

gtf_df = pd.read_pickle(output_path + "gene_info_data.pkl")


 ********************---process rna data---***************************** 



In [ ]:
from pdata.data_preprocess import rna_preprocess, atac_preprocess

adata_rna_file = output_path + "rna_processed.h5ad"

gene_info_file = output_path + "gene_info_filtered.pkl"

adata_rna, gene_info = rna_preprocess(adata_rna, gtf_df, adata_rna_file, gene_info_file)

## rna process

In [6]:
from pdata.data_preprocess import process_gene_info, filter_adata_rna, preprocess_adata_rna
adata_rna = ad.read_h5ad(output_path + "rna_origin.h5ad")

gtf_df = pd.read_pickle(output_path + "gene_info_data.pkl")

gtf_df = process_gene_info(gtf_df)

filter_gtf = gtf_df[gtf_df["gene_name"].isin(adata_rna.var_names)].copy()

adata_rna = filter_adata_rna(adata_rna)

common_gene =sorted(list( set(filter_gtf["gene_name"]) & set(adata_rna.var_names) ))

adata_rna = adata_rna[:, common_gene].copy()

adata_rna = preprocess_adata_rna(adata_rna, num_high_variable_gene=2000)

adata_rna = adata_rna[:, adata_rna.var['highly_variable']].copy()

filter_gtf = gtf_df[gtf_df["gene_name"].isin(adata_rna.var_names)].copy()

/home/wuyan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:283: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


In [7]:
adata_rna

AnnData object with n_obs × n_vars = 500 × 2000
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'n_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'mean', 'std'
    uns: 'hvg', 'log1p', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts', 'norm'
    obsp: 'distances', 'connectivities'

## filter rna

In [5]:
adata_rna = ad.read_h5ad(output_path + "rna_origin.h5ad")

data_type = 'sim'
gtf_df = pd.read_pickle(output_path + "gene_info_data.pkl")

adata_rna = adata_rna[:, ~adata_rna.var_names.str.startswith('ENSG00')].copy()

adata_rna.var_names_make_unique()

adata_rna.var['mt'] = adata_rna.var_names.str.startswith('MT-')

sc.pp.calculate_qc_metrics(adata_rna, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

MIN_GENES = 100
MAX_GENES = 6000   # 过滤掉基因数过高的细胞（可能是双细胞 Doublets）
MIN_COUNTS = 200


adata_rna = adata_rna[adata_rna.obs.n_genes_by_counts > MIN_GENES, :]
adata_rna = adata_rna[adata_rna.obs.n_genes_by_counts < MAX_GENES, :]
adata_rna = adata_rna[adata_rna.obs.total_counts > MIN_COUNTS, :]

if data_type == 'real':
    MAX_PCT_MT = 15.0  # 设定线粒体比例最高为 15%
    adata_rna = adata_rna[adata_rna.obs.pct_counts_mt < MAX_PCT_MT, :]

sc.pp.filter_genes(adata_rna, min_cells=10)

sc.pp.filter_genes(adata_rna, min_counts=100)

adata_rna = adata_rna[:, ~adata_rna.var['mt']].copy()

artifact_pattern = "^AC\d+|^AL\d+|^AP\d+|^AF\d+"

is_artifact = adata_rna.var_names.str.contains(artifact_pattern, regex=True)

adata_rna = adata_rna[:, ~is_artifact].copy()

adata_rna

/home/wuyan/conda_env/singlecellpreprocess/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:283: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


AnnData object with n_obs × n_vars = 500 × 8679
    obs: 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'n_counts'

In [ ]:
adata_rna = filter_adata_rna(adata_rna)